In [0]:
#retomando
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from pyspark.sql import functions as F

CATALOGO, SCHEMA = "workspace", "default"

## CRIANDO A SILVER_SCADA
CAMADA SILVER PARA OS RESULTADOS DO SUPERVISÓRIO

In [0]:
#criando a camada silver_scada
#mapeando colunas

mapa = [
    # nome na bronze | nome na silver | tipo de dados
    ("date_and_time",                          "tempo",                          "timestamp"),
    ("wind_speed_m_s",                          "wind_speed_m_s",                 "double"),
    ("wind_speed_standard_deviation_m_s",       "wind_speed_std_m_s",             "double"),
    ("wind_speed_minimum_m_s",                  "wind_speed_min_m_s",             "double"),
    ("wind_speed_maximum_m_s",                  "wind_speed_max_m_s",             "double"),
    ("power_kw",                                "power_kw",                       "double"),
    ("rotor_speed_rpm",                         "rotor_speed_rpm",                "double"),
    ("generator_rpm_rpm",                       "generator_rpm",                  "double"),
    ("blade_angle_pitch_position_a_°",          "blade_pitch_a_deg",              "double"),
    ("front_bearing_temperature_°c",            "front_bearing_temp_c",           "double"),
    ("rear_bearing_temperature_°c",             "rear_bearing_temp_c",            "double"),
    ("gear_oil_temperature_°c",                 "gear_oil_temp_c",                "double"),
    ("generator_bearing_front_temperature_°c",  "generator_bearing_front_temp_c", "double"),
    ("nacelle_ambient_temperature_°c",          "nacelle_ambient_temp_c",         "double"),
]

#apenas me preocupei no mapa de colunas que serão utilizadas para responder as perguntas. as demais colunas serão dropadas.

#leitura da tabela raw
bronze = spark.table(f"{CATALOGO}.{SCHEMA}.bronze_scada")

# começo a lista de colunas definindo chave primaria da turbina como numero int
colunas_silver = [F.col("turbina").cast("int").alias("turbina_id")]

#adicionando coluna do mapa convertida e transformada
for nome_bronze, nome_silver, tipo in mapa:
    coluna=F.col(nome_bronze).cast(tipo).alias(nome_silver)
    colunas_silver.append(coluna)

#aplicando e gravando
silver = bronze.select(colunas_silver)
(silver.write.mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOGO}.{SCHEMA}.silver_scada"))

print("ok, dados gravados na silver")

ok, dados gravados na silver


## VALIDANDO A SILVER_SCADA

In [0]:
silver = spark.table(f"{CATALOGO}.{SCHEMA}.silver_scada")

#primeiro teste: deve aparecer timestamp e double, sem str
silver.printSchema()

#segundo teste: deve aparecer amostra de 5 dados limpos
display(silver.limit(5)) 

#terceiro teste: verificando nulos na coluna de potencia (muito importante para o sentido do projeto)
total = silver.count()
nulos = silver.filter(F.col("power_kw").isNull()).count()
print(f"linhas: {total} | power_kw nulos: {nulos} | %: {round(nulos/total*100,2)}")



root
 |-- turbina_id: integer (nullable = true)
 |-- tempo: timestamp (nullable = true)
 |-- wind_speed_m_s: double (nullable = true)
 |-- wind_speed_std_m_s: double (nullable = true)
 |-- wind_speed_min_m_s: double (nullable = true)
 |-- wind_speed_max_m_s: double (nullable = true)
 |-- power_kw: double (nullable = true)
 |-- rotor_speed_rpm: double (nullable = true)
 |-- generator_rpm: double (nullable = true)
 |-- blade_pitch_a_deg: double (nullable = true)
 |-- front_bearing_temp_c: double (nullable = true)
 |-- rear_bearing_temp_c: double (nullable = true)
 |-- gear_oil_temp_c: double (nullable = true)
 |-- generator_bearing_front_temp_c: double (nullable = true)
 |-- nacelle_ambient_temp_c: double (nullable = true)



turbina_id,tempo,wind_speed_m_s,wind_speed_std_m_s,wind_speed_min_m_s,wind_speed_max_m_s,power_kw,rotor_speed_rpm,generator_rpm,blade_pitch_a_deg,front_bearing_temp_c,rear_bearing_temp_c,gear_oil_temp_c,generator_bearing_front_temp_c,nacelle_ambient_temp_c
1,2018-05-17T21:00:00.000Z,2.9048526287078857,0.4581696391105652,2.0159499645233154,3.861062526702881,17.462011337280273,8.151208877563477,968.1171875,1.4295004606246948,59.932498931884766,63.275001525878906,57.09000015258789,42.592498779296875,9.399999618530273
1,2018-05-17T21:10:00.000Z,3.0757017135620117,0.3271157145500183,2.2461252212524414,3.686203718185425,25.05955696105957,8.140153884887695,967.8040771484375,1.3400005102157593,58.95000076293945,62.47999954223633,56.54999923706055,40.85499954223633,9.0024995803833
1,2018-05-17T21:20:00.000Z,3.4196462631225586,0.49594902992248535,2.0842602252960205,4.062651634216309,44.25772476196289,8.130706787109375,967.907470703125,0.9900000095367432,59.36249923706055,62.51750183105469,56.2599983215332,37.48249816894531,8.822500228881836
1,2018-05-17T21:30:00.000Z,3.287592649459839,0.35808223485946655,2.566884994506836,3.9587016105651855,38.56118392944336,8.142118453979492,968.3922119140625,1.0709998607635498,58.994998931884766,62.092498779296875,55.994998931884766,35.88999938964844,8.837499618530273
1,2018-05-17T21:40:00.000Z,2.9371883869171143,0.5469922423362732,1.5533725023269653,3.9787490367889404,28.33646011352539,8.139389038085938,968.8828735351562,1.2675020694732666,58.907501220703125,61.98500061035156,55.72500228881836,36.70499801635742,8.78499984741211


linhas: 315360 | power_kw nulos: 11002 | %: 3.49


a tipagem deu certo, como esperado, com a quantidade de linhas esperada. 
o 3,5% aproximado de nulos para potencia evidencia que na transformação de str para double, texto vazio ou invalido ficou nulo. pode ser achado de parada de manutenção, ou falha inesperada. fica de ponto de atenção para avaliar nas futuras querys.

## CRIANDO A SILVER STATUS
detalhe importante que na status, o timestamp end and duration pode ter eventos informativos/instantaneos que, ao converter de string pra timestamp, pode retornar nulos. Por isso prefiro usar o try_cast ao invés do cast, nesse caso, pegando emprestado o SQL rapidinho.

In [0]:
# retomando os nomes das colunas da bronze_status
print(spark.table(f"{CATALOGO}.{SCHEMA}.bronze_status").columns)


['timestamp_start', 'timestamp_end', 'duration', 'status', 'code', 'message', 'comment', 'service_contract_category', 'iec_category', 'turbina', 'arquivo_origem']


In [0]:
mapa_status = [
    ("timestamp_start",            "tempo_inicio",       "timestamp"),
    ("timestamp_end",              "tempo_fim",          "timestamp"),
    ("status",                     "status",             "string"),
    ("code",                       "codigo",             "string"),
    ("message",                    "mensagem",           "string"),
    ("service_contract_category",  "categoria_contrato", "string"),
    ("iec_category",               "categoria_iec",      "string"),
]

bronze_st = spark.table(f"{CATALOGO}.{SCHEMA}.bronze_status")

# a chave da turbina, tolerando lixo
colunas = [ F.expr("try_cast(turbina AS int)").alias("turbina_id") ]

# cada coluna do mapa, com try_cast via SQL
for nome_bronze, nome_silver, tipo in mapa_status:
    expressao = f"try_cast({nome_bronze} AS {tipo})"
    colunas.append(F.expr(expressao).alias(nome_silver))

silver_st = bronze_st.select(colunas)

(silver_st.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.{SCHEMA}.silver_status"))

print("silver_status criada")

silver_status criada


## VALIDANDO A SILVER_STATUS

In [0]:
silver_st = spark.table(f"{CATALOGO}.{SCHEMA}.silver_status")

# 1) tipos: tempo_inicio e tempo_fim devem estar como timestamp
silver_st.printSchema()

# 2) amostra dos eventos já limpos
display(silver_st.limit(5))

# 3) quantos eventos por turbina + quantos sem fim registrado
print("eventos por turbina:")
display(silver_st.groupBy("turbina_id").count().orderBy("turbina_id"))

total = silver_st.count()
sem_fim = silver_st.filter(F.col("tempo_fim").isNull()).count()
print(f"total de eventos: {total}  |  sem tempo_fim: {sem_fim}")

root
 |-- turbina_id: integer (nullable = true)
 |-- tempo_inicio: timestamp (nullable = true)
 |-- tempo_fim: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- mensagem: string (nullable = true)
 |-- categoria_contrato: string (nullable = true)
 |-- categoria_iec: string (nullable = true)



turbina_id,tempo_inicio,tempo_fim,status,codigo,mensagem,categoria_contrato,categoria_iec
1,2018-01-03T02:23:56.000Z,2018-01-03T02:50:55.000Z,Stop,692,Pitch run-away (hub box v.>=4),Controller error of the WP3100 (16),Forced outage
1,2018-01-03T02:23:56.000Z,null,Informational,100035,Brake program 170,null,Full Performance
1,2018-01-03T02:34:12.000Z,null,Informational,100060,Brake program 52,null,Full Performance
1,2018-01-03T02:50:55.000Z,null,Informational,0,System OK,System OK (32),Full Performance
1,2018-01-03T02:50:55.000Z,null,Informational,100110,Bypass limit switches,null,Full Performance


eventos por turbina:


turbina_id,count
1,10674
2,9443
3,12515
4,9933
5,11444
6,15954


total de eventos: 69963  |  sem tempo_fim: 60301


In [0]:
silver_st = spark.table(f"{CATALOGO}.{SCHEMA}.silver_status")

display(
    silver_st.groupBy("status")
        .agg(
            F.count("*").alias("total"),
            F.sum(F.when(F.col("tempo_fim").isNull(), 1).otherwise(0)).alias("sem_fim")
        )
        .orderBy(F.desc("total"))
)

status,total,sem_fim
Informational,68608,60301
Stop,733,0
Warning,524,0
Communication,98,0


a tipagem deu certo, mas os tempos sem fim são muitos.

isso acontece por causa dos eventos instântaneos tipo "brake program", "system ok", "bypass limit switches" etc. não tem fim porque não duram, são instatâneos e representam um marco.

esses dados instântaneos podem virar outra análise bônus, pois na reconciliação será usado apenas dados que possuem janela (duração), e não os instântaneos informativos.

esses nulos serão filtrados, não corrigidos, não tem nada pra corrigir.

## CARREGANDO CADASTRO DAS TURBINAS
avaliando o estático Kelmarsh_WT_static.csv

In [0]:
#primeiro avalio o arquivo cru pra saber o que vou fazer

VOL = f"/Volumes/{CATALOGO}/{SCHEMA}/kelmarsh_raw"

caminho_static = f"{VOL}/ref/Kelmarsh_WT_static.csv"
# mostra as primeiras linhas do jeito que estão no arquivo
with open(caminho_static, encoding="utf-8", errors="replace") as f:
    for i in range(8):
        print(i, repr(f.readline()))

0 '\ufeffWind Farm,Title,Alternative Title,Identity,Manufacturer,Model,Rated power (kW),Hub Height (m),Rotor Diameter (m),Latitude,Longitude,Elevation (m),Country,Commercial Operations Date\n'
1 'Kelmarsh,Kelmarsh 1,KWF1,SEN 93420,Senvion,MM92,2050,78.5,92,52.400604,-0.947133,145.598,UK,15/04/2016\n'
2 'Kelmarsh,Kelmarsh 2,KWF2,SEN 93421,Senvion,MM92,2050,78.5,92,52.402551,-0.949527,156.577,UK,15/04/2016\n'
3 'Kelmarsh,Kelmarsh 3,KWF3,SEN 93422,Senvion,MM92,2050,68.5,92,52.403834,-0.94419,153.477,UK,15/04/2016\n'
4 'Kelmarsh,Kelmarsh 4,KWF4,SEN 93423,Senvion,MM92,2050,78.5,92,52.398781,-0.94115,146.313,UK,15/04/2016\n'
5 'Kelmarsh,Kelmarsh 5,KWF5,SEN 93424,Senvion,MM92,2050,78.5,92,52.402308,-0.940537,142.901,UK,15/04/2016\n'
6 'Kelmarsh,Kelmarsh 6,KWF6,SEN 93425,Senvion,MM92,2050,68.5,92,52.400687,-0.936093,135.039,UK,15/04/2016\n'
7 ''


In [0]:
df_stc = pd.read_csv(caminho_static, sep=",", encoding="utf-8-sig")

#Trazendo de volta a funcao limpar nome da bronze
def limpar_nome(nome):
    nome = nome.strip().lower()
    for c in r" ,;(){}/\-.#:%º°ª<>":
        nome = nome.replace(c, "_")
    while "__" in nome:
        nome = nome.replace("__", "_")
    return nome.strip("_")

df_stc.columns = [limpar_nome(c) for c in df_stc.columns]

print("colunas:", df_stc.columns.tolist())
df_stc

colunas: ['wind_farm', 'title', 'alternative_title', 'identity', 'manufacturer', 'model', 'rated_power_kw', 'hub_height_m', 'rotor_diameter_m', 'latitude', 'longitude', 'elevation_m', 'country', 'commercial_operations_date']


,wind_farm,title,alternative_title,identity,manufacturer,model,rated_power_kw,hub_height_m,rotor_diameter_m,latitude,longitude,elevation_m,country,commercial_operations_date
0,Kelmarsh,Kelmarsh 1,KWF1,SEN 93420,Senvion,MM92,2050,78.5,92,52.400604,-0.947133,145.598,UK,15/04/2016
1,Kelmarsh,Kelmarsh 2,KWF2,SEN 93421,Senvion,MM92,2050,78.5,92,52.402551,-0.949527,156.577,UK,15/04/2016
2,Kelmarsh,Kelmarsh 3,KWF3,SEN 93422,Senvion,MM92,2050,68.5,92,52.403834,-0.944190,153.477,UK,15/04/2016
3,Kelmarsh,Kelmarsh 4,KWF4,SEN 93423,Senvion,MM92,2050,78.5,92,52.398781,-0.941150,146.313,UK,15/04/2016
4,Kelmarsh,Kelmarsh 5,KWF5,SEN 93424,Senvion,MM92,2050,78.5,92,52.402308,-0.940537,142.901,UK,15/04/2016
5,Kelmarsh,Kelmarsh 6,KWF6,SEN 93425,Senvion,MM92,2050,68.5,92,52.400687,-0.936093,135.039,UK,15/04/2016


ok, parto para montar e gravar a silver para turbina

escolho as colunas úteis, tipo e renomeio pro padrão do catálogo. A chave: Identity (tipo SEN 93420) é o código do fabricante, não  turbina_id (1–6). O nosso número está no fim do Title ("Kelmarsh 1"). Então extraio o número dali:

In [0]:
# vira dataframe Spark pra trabalhar
sdf = spark.createDataFrame(df_stc)

silver_turbina = sdf.select(
    # turbina_id = último pedaço do "title" (Kelmarsh 1 -> 1)
    F.expr("try_cast(split(title, ' ')[1] AS int)").alias("turbina_id"),
    F.col("title").alias("nome"),
    F.col("latitude").cast("double").alias("latitude"),
    F.col("longitude").cast("double").alias("longitude"),
    F.col("rated_power_kw").cast("int").alias("potencia_nominal_kw"),
    F.col("rotor_diameter_m").cast("double").alias("diametro_rotor_m"),
    F.col("hub_height_m").cast("double").alias("altura_cubo_m"),
)

(silver_turbina.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.{SCHEMA}.silver_turbina"))

display(silver_turbina.orderBy("turbina_id"))

turbina_id,nome,latitude,longitude,potencia_nominal_kw,diametro_rotor_m,altura_cubo_m
1,Kelmarsh 1,52.400604,-0.947133,2050,92.0,78.5
2,Kelmarsh 2,52.402551,-0.949527,2050,92.0,78.5
3,Kelmarsh 3,52.403834,-0.94419,2050,92.0,68.5
4,Kelmarsh 4,52.398781,-0.94115,2050,92.0,78.5
5,Kelmarsh 5,52.402308,-0.940537,2050,92.0,78.5
6,Kelmarsh 6,52.400687,-0.936093,2050,92.0,68.5
